In [ ]:
# Bước 1: Clone source
!git clone https://github.com/sangtran0897/OmniVoice.git
%cd OmniVoice
!pip install -e .

# Bước 2: Trỏ Python tới source folder
import sys
sys.path.insert(0, '/content/omnivoice')



In [ ]:
# Bước 3: Code y chang như cũ
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "kjanh/KhanhTTS-OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading reference audio from Hugging Face...")

ref_audio_path = hf_hub_download(
    repo_id="sangtran0897/omnivoice-clone",
    filename="xkt.WAV",
    repo_type="dataset",   # quan trọng: repo này là dataset
    token=''
)


print(f"Loaded: {ref_audio_path}")


In [ ]:
!git pull

In [ ]:
!git pull
import os
import re
import subprocess
import tempfile

import ipywidgets as widgets
import numpy as np
import soundfile as sf

from IPython.display import (
    display,
    Audio,
    clear_output,
    HTML,
    Javascript,
)

from japanese_romaji_words import JAPANESE_ROMAJI_WORDS


# =========================
# Cau hinh
# =========================

SAMPLE_RATE = 24000

# Khoang nghi that duoc chen giua cac cau.
nghi_cau = 0.25

# Tang toc hau ky cho tung doan audio.
segment_speed = 1.05

# He so rut thoi luong model generate.
# Tang qua cao co the lam bo chu hoac bo ten rieng.
duration_speed = 1.2

ref_text = (
    "Hi, "
    "xin chào các con vợ, "
    "chúng ta đang ở đây với nội dung nóng hổi và được mong đợi nhất trong tuần."
)


# =========================
# Cau hinh ten Nhat romaji
# =========================

# Moi lan mot tu romaji xuat hien se duoc cong them
# mot khoang duration nho.
#
# Cac tu nay da duoc tinh trong speaking units va complex buffer,
# nen khong can cong qua cao.
JAPANESE_ROMAJI_EXTRA_PER_WORD = 0.055

# Gioi han tong duration bo sung trong mot cau.
JAPANESE_ROMAJI_MAX_EXTRA = 0.45

# Khoang cong them vao duration toi thieu cho moi tu romaji.
JAPANESE_ROMAJI_MINIMUM_PER_WORD = 0.065


# =========================
# Ham xu ly audio
# =========================

def speedup_audio(input_path, output_path, speed=1.08):
    """Tang toc mot file audio bang FFmpeg."""
    atempo_filter = build_atempo_filter(speed)

    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            input_path,
            "-filter:a",
            atempo_filter,
            "-ar",
            str(SAMPLE_RATE),
            "-ac",
            "1",
            "-vn",
            output_path,
        ],
        check=True,
    )


def build_atempo_filter(speed):
    """Tao chuoi bo loc atempo hop le cho FFmpeg."""
    speed = float(speed)

    if speed <= 0:
        raise ValueError("Toc do audio phai lon hon 0.")

    filters = []

    while speed > 2.0:
        filters.append("atempo=2.0")
        speed /= 2.0

    while speed < 0.5:
        filters.append("atempo=0.5")
        speed /= 0.5

    filters.append(f"atempo={speed:.6f}")
    return ",".join(filters)


def speedup_audio_segment(
    audio_array,
    speed=1.10,
    sample_rate=24000,
):
    """Tang toc rieng mot doan audio sau khi model generate."""
    audio_array = normalize_audio_array(audio_array)

    if speed is None or abs(float(speed) - 1.0) < 0.0001:
        return audio_array

    input_temp_path = None
    output_temp_path = None

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".wav",
            delete=False,
        ) as input_temp:
            input_temp_path = input_temp.name

        with tempfile.NamedTemporaryFile(
            suffix=".wav",
            delete=False,
        ) as output_temp:
            output_temp_path = output_temp.name

        sf.write(
            input_temp_path,
            audio_array,
            sample_rate,
        )

        atempo_filter = build_atempo_filter(speed)

        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-loglevel",
                "error",
                "-i",
                input_temp_path,
                "-filter:a",
                atempo_filter,
                "-ar",
                str(sample_rate),
                "-ac",
                "1",
                "-vn",
                output_temp_path,
            ],
            check=True,
        )

        processed_audio, processed_sr = sf.read(
            output_temp_path,
            dtype="float32",
        )

        processed_audio = normalize_audio_array(processed_audio)

        if processed_sr != sample_rate:
            raise ValueError(
                f"Sample rate sau khi tang toc la "
                f"{processed_sr}, can {sample_rate}."
            )

        return processed_audio

    finally:
        if (
            input_temp_path
            and os.path.exists(input_temp_path)
        ):
            os.remove(input_temp_path)

        if (
            output_temp_path
            and os.path.exists(output_temp_path)
        ):
            os.remove(output_temp_path)


def make_silence(
    seconds=0.18,
    sample_rate=24000,
):
    """Tao khoang lang giua cac cau."""
    return np.zeros(
        int(seconds * sample_rate),
        dtype=np.float32,
    )


def normalize_audio_array(audio_array):
    """Dam bao audio la numpy array mot chieu float32."""
    audio_array = np.asarray(audio_array)
    audio_array = np.squeeze(audio_array)

    if audio_array.ndim > 1:
        if audio_array.shape[-1] <= 8:
            audio_array = np.mean(
                audio_array,
                axis=-1,
            )
        elif audio_array.shape[0] <= 8:
            audio_array = np.mean(
                audio_array,
                axis=0,
            )
        else:
            audio_array = audio_array.reshape(-1)

    return audio_array.astype(np.float32)


# =========================
# Mau ky tu phan tich tu
# =========================

VIETNAMESE_MARK_PATTERN = (
    r"[aàáảãạăằắẳẵặâầấẩẫậ"
    r"eèéẻẽẹêềếểễệ"
    r"iìíỉĩị"
    r"oòóỏõọôồốổỗộơờớởỡợ"
    r"uùúủũụưừứửữự"
    r"yỳýỷỹỵđ"
    r"AÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬ"
    r"EÈÉẺẼẸÊỀẾỂỄỆ"
    r"IÌÍỈĨỊ"
    r"OÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢ"
    r"UÙÚỦŨỤƯỪỨỬỮỰ"
    r"YỲÝỶỸỴĐ]"
)

# Mau nay chi nhan cac ky tu dac trung cua tieng Viet.
VIETNAMESE_DIACRITIC_PATTERN = (
    r"[àáảãạăằắẳẵặâầấẩẫậ"
    r"èéẻẽẹêềếểễệ"
    r"ìíỉĩị"
    r"òóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữự"
    r"ỳýỷỹỵđ"
    r"ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬ"
    r"ÈÉẺẼẸÊỀẾỂỄỆ"
    r"ÌÍỈĨỊ"
    r"ÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢ"
    r"ÙÚỦŨỤƯỪỨỬỮỰ"
    r"ỲÝỶỸỴĐ]"
)

VOWEL_PATTERN = (
    r"[aeiouy"
    r"àáảãạăằắẳẵặâầấẩẫậ"
    r"èéẻẽẹêềếểễệ"
    r"ìíỉĩị"
    r"òóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữự"
    r"ỳýỷỹỵ]+"
)

CONSONANT_CLUSTER_PATTERN = (
    r"[^aeiouy"
    r"àáảãạăằắẳẵặâầấẩẫậ"
    r"èéẻẽẹêềếểễệ"
    r"ìíỉĩị"
    r"òóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữự"
    r"ỳýỷỹỵ0-9]{2,}"
)


# =========================
# Tu tieng Viet khong dau thong dung
# =========================

# Danh sach nay khong co muc tieu bao phu toan bo tieng Viet.
# No chi loai tru nhung tu khong dau thong dung de chung khong bi
# tinh nham thanh tu nuoc ngoai hoac tu phuc tap.
#
# De bao ve ten rieng:
# - Tu viet hoa o giua cau van duoc xem xet nhu ten rieng.
# - Tu trong danh sach chi duoc mien tinh phuc tap khi viet thuong,
#   hoac nam o dau cau do quy tac viet hoa thong thuong.
VIETNAMESE_UNMARKED_COMMON_WORDS = {
    # Dai tu, tu xung ho, chi nguoi
    "ai", "anh", "ba", "ban", "bo", "bac", "be", "chi", "chu", "co",
    "con", "em", "ho", "minh", "nguoi", "ong", "ta", "toi", "tui",
    "may", "no", "chung", "chungta", "chungtoi", "chungminh",

    # Tu noi, quan he, gioi tu
    "va", "voi", "cung", "cua", "cho", "do", "tai", "tu", "den", "ve",
    "theo", "tren", "duoi", "trong", "ngoai", "giua", "sau", "truoc",
    "ben", "canh", "qua", "bang", "nhu", "neu", "thi", "ma", "nhung",
    "tuy", "du", "vi", "nen", "hay", "hoac", "con", "roi", "lai",
    "dongthoi", "trongkhi", "saukhi", "truockhi", "boivi", "macdu",

    # Tu chi dinh, luong, muc do
    "nay", "kia", "do", "day", "ay", "moi", "mot", "hai", "ba", "bon",
    "nam", "sau", "bay", "tam", "chin", "muoi", "vai", "nhieu", "it",
    "tatca", "toanbo", "phanlon", "moi", "tung", "moi", "deu", "rat",
    "kha", "qua", "lam", "hon", "nhat", "gan", "xa", "them", "bot",

    # Thoi gian, trinh tu
    "ngay", "dem", "sang", "trua", "chieu", "toi", "hom", "nay", "mai",
    "qua", "truoc", "sau", "luc", "khi", "gio", "phut", "giay", "lan",
    "dau", "cuoi", "tieptheo", "saucuong", "bandau", "hientai", "truocdo",

    # Dong tu thong dung
    "la", "co", "khong", "duoc", "bi", "lam", "noi", "nghe", "thay",
    "nhin", "biet", "hieu", "nghi", "nho", "quen", "muon", "can", "phai",
    "nen", "the", "giu", "dua", "lay", "mang", "dem", "cho", "goi", "hoi",
    "dap", "doc", "viet", "xem", "tim", "gap", "den", "di", "ve", "ra",
    "vao", "len", "xuong", "qua", "lai", "dung", "ngoi", "nam", "chay",
    "bay", "roi", "roi", "mat", "con", "song", "chet", "xuat", "hien",
    "bien", "mat", "thaydoi", "batdau", "ketthuc", "tieptuc", "trothanh",
    "nhanra", "chothay", "taora", "xayra", "nhacden", "lienquan",

    # Tinh tu, trang tu thong dung
    "lon", "nho", "cao", "thap", "dai", "ngan", "nhanh", "cham", "manh",
    "yeu", "moi", "cu", "tot", "xau", "dung", "sai", "ro", "that", "gia",
    "de", "kho", "don", "gian", "dacbiet", "quantrong", "nghiemtrong",
    "nhanhchong", "hoantoan", "thucte", "tudong", "tructiep", "gant nhu",

    # Danh tu thong dung
    "viec", "chuyen", "dieu", "thu", "cach", "lydo", "ketqua", "thongtin",
    "noidung", "vanban", "cau", "tu", "ten", "tieng", "giong", "amthanh",
    "thoigian", "tocdo", "khoang", "phan", "doan", "file", "model", "audio",
    "nhanvat", "tinhhuong", "van de", "cauhoi", "traloi", "thegioi",
    "sucmanh", "khanang", "muctieu", "truonghop", "quatrinh", "he thong",
    "phuongphap",

    # Tu thuong dung trong van ke, binh luan
    "vay", "vaynen", "tuy nhien", "don gian", "ro rang", "that ra",
    "luc nay", "cungluc", "ngaylap tuc", "cuoicung", "chuahet",
    "dangchu y", "dangnoi", "co the", "khongthe", "duongnhien",
    "chinhxac", "gan nhu", "it nhat",
}

# Loai bo khoang trang vo tinh trong cac muc ghep.
VIETNAMESE_UNMARKED_COMMON_WORDS = {
    word.replace(" ", "")
    for word in VIETNAMESE_UNMARKED_COMMON_WORDS
}


# =========================
# Ham xu ly text
# =========================

def extract_words(text):
    """Tach tu tieng Viet, tu nuoc ngoai va chu so."""
    return re.findall(
        r"[A-Za-zÀ-ỹĐđ0-9]+",
        text,
    )


def count_vietnamese_units(text):
    """Dem so tu thuc te trong cau."""
    return len(extract_words(text))


def normalize_word(word):
    """Chuyen tu ve chu thuong de phan tich."""
    return word.strip().lower()


def contains_vietnamese_marks(word):
    """Kiem tra tu co ky tu dac trung cua tieng Viet hay khong."""
    return bool(
        re.search(
            VIETNAMESE_DIACRITIC_PATTERN,
            word,
        )
    )


def is_japanese_romaji_word(word):
    """
    Kiem tra tu co nam trong danh sach ten Nhat romaji hay khong.

    Phan nay chi la mot lop bo sung nho, khong thay doi
    cach phan loai tieng Viet va tieng Anh hien tai.
    """
    normalized = normalize_word(word)

    if not normalized:
        return False

    return normalized in JAPANESE_ROMAJI_WORDS


def count_japanese_romaji_words(text):
    """
    Dem so lan cac ten Nhat romaji xuat hien trong cau.

    Mot tu lap lai nhieu lan van duoc tinh theo so lan xuat hien
    vi model van phai phat am moi lan.
    """
    words = extract_words(text)

    return sum(
        1
        for word in words
        if is_japanese_romaji_word(word)
    )


def estimate_japanese_romaji_buffer(text):
    """
    Cong mot khoang duration nho cho ten Nhat viet bang chu Latin.

    Day la phan bo sung. Cac tu nay van duoc tinh binh thuong
    trong calculate_speaking_units va estimate_complex_word_buffer.
    """
    romaji_count = count_japanese_romaji_words(text)

    if romaji_count == 0:
        return 0.0

    return min(
        romaji_count * JAPANESE_ROMAJI_EXTRA_PER_WORD,
        JAPANESE_ROMAJI_MAX_EXTRA,
    )


def is_common_unmarked_vietnamese_word(
    word,
    word_index=0,
):
    """
    Kiem tra mot tu khong dau co nam trong nhom
    tieng Viet thong dung hay khong.

    Tu viet hoa o giua cau khong duoc loai tru,
    vi no co kha nang la ten rieng.

    Vi du: "Long" trong "gap Long" van duoc bao ve nhu ten.
    """
    normalized = normalize_word(word)

    if normalized not in VIETNAMESE_UNMARKED_COMMON_WORDS:
        return False

    if (
        word_index > 0
        and is_capitalized_word(word)
    ):
        return False

    return True


def count_vowel_groups(word):
    """Dem so cum nguyen am trong mot tu."""
    normalized = normalize_word(word)

    return len(
        re.findall(
            VOWEL_PATTERN,
            normalized,
        )
    )


def count_consonant_clusters(word):
    """Dem so cum co tu hai phu am lien nhau."""
    normalized = normalize_word(word)

    return len(
        re.findall(
            CONSONANT_CLUSTER_PATTERN,
            normalized,
        )
    )


def is_capitalized_word(word):
    """Kiem tra tu co bat dau bang chu hoa hay khong."""
    if not word:
        return False

    return (
        word[0].isalpha()
        and word[0].isupper()
    )


def estimate_word_complexity_score(
    word,
    word_index=0,
):
    """Tinh diem phuc tap cua tu ma khong can danh sach ten rieng."""
    normalized = normalize_word(word)

    if not normalized or normalized.isdigit():
        return 0

    if contains_vietnamese_marks(word):
        return 0

    # Tu tieng Viet khong dau thong dung khong bi tinh nham
    # thanh tu nuoc ngoai hoac tu phuc tap.
    if is_common_unmarked_vietnamese_word(
        word,
        word_index,
    ):
        return 0

    word_length = len(normalized)
    vowel_group_count = count_vowel_groups(word)
    consonant_cluster_count = count_consonant_clusters(word)

    score = 0

    if word_length >= 4:
        score += 1

    if word_length >= 6:
        score += 1

    if word_length >= 8:
        score += 1

    if (
        word_index > 0
        and is_capitalized_word(word)
    ):
        score += 2

    if (
        word_index == 0
        and is_capitalized_word(word)
    ):
        score += 1

    if vowel_group_count >= 2:
        score += 1

    if vowel_group_count >= 3:
        score += 1

    if consonant_cluster_count >= 1:
        score += 1

    if consonant_cluster_count >= 2:
        score += 1

    return score


def is_likely_complex_word(
    word,
    word_index=0,
):
    """Xac dinh tuong doi mot tu co cach phat am phuc tap."""
    return (
        estimate_word_complexity_score(
            word,
            word_index,
        )
        >= 2
    )


def estimate_word_weight(
    word,
    word_index=0,
):
    """Uoc tinh trong so phat am cho tung tu."""
    normalized = normalize_word(word)

    if not normalized:
        return 0.0

    if normalized.isdigit():
        if len(normalized) <= 2:
            return 1.0

        return min(
            1.0 + len(normalized) * 0.05,
            1.40,
        )

    if contains_vietnamese_marks(word):
        return 1.0

    # Tu tieng Viet khong dau thong dung dung trong so co ban.
    # Ten viet hoa giua cau van khong vao nhanh nay.
    if is_common_unmarked_vietnamese_word(
        word,
        word_index,
    ):
        return 1.0

    word_length = len(normalized)
    vowel_group_count = count_vowel_groups(word)
    consonant_cluster_count = count_consonant_clusters(word)

    weight = 1.0

    if word_length >= 4:
        weight += 0.06

    if word_length >= 6:
        weight += 0.10

    if word_length >= 8:
        weight += 0.10

    if vowel_group_count >= 2:
        weight += min(
            (vowel_group_count - 1) * 0.07,
            0.18,
        )

    if consonant_cluster_count >= 1:
        weight += min(
            consonant_cluster_count * 0.08,
            0.20,
        )

    if (
        word_index > 0
        and is_capitalized_word(word)
    ):
        weight += 0.12

    if (
        word_index == 0
        and is_capitalized_word(word)
    ):
        weight += 0.03

    complexity_score = estimate_word_complexity_score(
        word,
        word_index,
    )

    if complexity_score >= 4:
        weight += 0.06

    if complexity_score >= 6:
        weight += 0.05

    return min(
        weight,
        1.60,
    )


def calculate_speaking_units(text):
    """Tinh tong don vi phat am cua cau."""
    words = extract_words(text)

    if not words:
        return 1.0

    return max(
        sum(
            estimate_word_weight(
                word,
                word_index,
            )
            for word_index, word in enumerate(words)
        ),
        1.0,
    )


def count_likely_complex_words(text):
    """Dem so tu co kha nang can them thoi gian phat am."""
    words = extract_words(text)

    return sum(
        1
        for word_index, word in enumerate(words)
        if is_likely_complex_word(
            word,
            word_index,
        )
    )


def estimate_pause_seconds(text):
    """Tinh thoi gian bo sung dua theo dau cau."""
    comma = len(
        re.findall(
            r"[,，]",
            text,
        )
    ) * 0.09

    semi = len(
        re.findall(
            r"[;:；：]",
            text,
        )
    ) * 0.14

    end = len(
        re.findall(
            r"[.!?。！？]",
            text,
        )
    ) * 0.20

    newline = text.count("\n") * 0.25

    return (
        comma
        + semi
        + end
        + newline
    )


def estimate_short_pause_seconds(text):
    """Tinh thoi gian dau cau cho cau ngan tu ba chu tro xuong."""
    comma = len(
        re.findall(
            r"[,，]",
            text,
        )
    ) * 0.03

    semi = len(
        re.findall(
            r"[;:；：]",
            text,
        )
    ) * 0.05

    has_end_punctuation = bool(
        re.search(
            r"[.!?。！？]",
            text,
        )
    )

    end = (
        0.06
        if has_end_punctuation
        else 0.0
    )

    return (
        comma
        + semi
        + end
    )


def estimate_complex_word_buffer(text):
    """Cong khoang dem cho cau co tu phat am phuc tap."""
    complex_word_count = count_likely_complex_words(text)

    if complex_word_count == 0:
        return 0.0

    return min(
        complex_word_count * 0.06,
        0.24,
    )


def estimate_duration_from_ref(
    text,
    ref_audio_path,
    ref_text,
    speed=1.0,
):
    """
    Uoc tinh thoi luong model can de tao cau.

    Phan tieng Viet va tieng Anh duoc giu nguyen.
    Ten Nhat romaji chi duoc cong them mot lop duration nho.
    """
    ref_audio, sr = sf.read(ref_audio_path)

    ref_duration = len(ref_audio) / sr

    ref_units = max(
        count_vietnamese_units(ref_text),
        1,
    )

    target_word_count = max(
        count_vietnamese_units(text),
        1,
    )

    target_speaking_units = max(
        calculate_speaking_units(text),
        1.0,
    )

    complex_word_count = count_likely_complex_words(text)
    japanese_romaji_count = count_japanese_romaji_words(text)

    sec_per_unit = ref_duration / ref_units

    sec_per_unit = min(
        max(sec_per_unit, 0.22),
        0.40,
    )

    # Giu nguyen nhanh xu ly cau ngan.
    #
    # Neu cau ngan co ten Nhat romaji, khong dung nhanh nay
    # vi short_max_duration co the gioi han duration qua som.
    if (
        target_word_count <= 3
        and complex_word_count == 0
        and japanese_romaji_count == 0
    ):
        short_sec_per_unit = min(
            sec_per_unit,
            0.23,
        )

        duration = (
            target_word_count
            * short_sec_per_unit
        )

        duration += estimate_short_pause_seconds(text)
        duration /= speed

        short_min_duration = {
            1: 0.36,
            2: 0.46,
            3: 0.60,
        }

        short_max_duration = {
            1: 0.54,
            2: 0.68,
            3: 0.82,
        }

        duration = max(
            duration,
            short_min_duration[target_word_count],
        )

        duration = min(
            duration,
            short_max_duration[target_word_count],
        )

        return round(
            duration,
            2,
        )

    duration = (
        target_speaking_units
        * sec_per_unit
    )

    duration += estimate_pause_seconds(text)
    duration += estimate_complex_word_buffer(text)

    # Lop duration bo sung rieng cho ten Nhat romaji.
    #
    # Vi cac tu nay da duoc tinh trong speaking units
    # va complex word buffer, chi cong them mot luong nho.
    duration += estimate_japanese_romaji_buffer(text)

    duration /= speed

    base_minimum_duration = (
        target_word_count
        * 0.20
    )

    if complex_word_count > 0:
        complex_minimum_duration = (
            target_word_count * 0.21
            + complex_word_count * 0.05
        )

        minimum_duration = max(
            1.05,
            base_minimum_duration,
            complex_minimum_duration,
        )
    else:
        minimum_duration = max(
            1.05,
            base_minimum_duration,
        )

    # Muc san bo sung cho cau co ten Nhat romaji.
    #
    # Muc san nay chi co tac dung khi duration chung
    # dang bi tinh qua thap.
    if japanese_romaji_count > 0:
        japanese_romaji_minimum = (
            target_word_count * 0.20
            + japanese_romaji_count
            * JAPANESE_ROMAJI_MINIMUM_PER_WORD
        )

        minimum_duration = max(
            minimum_duration,
            japanese_romaji_minimum,
        )

    return round(
        max(
            duration,
            minimum_duration,
        ),
        2,
    )


def split_text_to_sentences(text):
    """Cat text thanh tung cau va giu lai dau cau cuoi."""
    text = re.sub(
        r"\s+",
        " ",
        text.strip(),
    )

    sentences = re.findall(
        r"[^.!?。！？]+[.!?。！？]+|[^.!?。！？]+$",
        text,
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# =========================
# UI
# =========================

output = widgets.Output()

text_box = widgets.Textarea(
    value="",
    placeholder="Paste doan text vao day...",
    description="Text:",
    layout=widgets.Layout(
        width="100%",
        height="300px",
    ),
    style={
        "description_width": "50px",
    },
)

scroll_lock_button = widgets.Button(
    description="Mo khoa cuon",
    button_style="info",
    icon="unlock",
    tooltip="Mo khoa de cuon man hinh binh thuong",
    layout=widgets.Layout(
        width="180px",
        height="38px",
    ),
)

lock_status = widgets.HTML(
    value=(
        "<span style='color:#d32f2f; font-weight:600;'>"
        "Man hinh dang duoc khoa tai khung nhap."
        "</span>"
    )
)

scroll_lock_state = {
    "locked": True,
}

lock_anchor = HTML(
    """
<div
    id="omnivoice-input-anchor"
    style="
        display: block;
        width: 100%;
        height: 1px;
        margin: 0;
        padding: 0;
        scroll-margin-top: 10px;
    "
></div>
"""
)


# =========================
# Generate
# =========================

def on_text_change(change):
    if change["name"] != "value":
        return

    input_text = change["new"].strip()

    if not input_text:
        return

    with output:
        clear_output(wait=True)
        print("Dang generate...")

        sentences = split_text_to_sentences(input_text)

        if not sentences:
            print("Khong co cau nao de generate.")
            return

        print(f"Tong so cau: {len(sentences)}")
        print(f"Nghi giua moi cau: {nghi_cau}s")
        print(
            f"Toc do rieng cua tung doan: "
            f"{segment_speed}x"
        )

        all_audio_segments = []

        for idx, sentence in enumerate(
            sentences,
            start=1,
        ):
            duration = estimate_duration_from_ref(
                sentence,
                ref_audio_path,
                ref_text,
                speed=duration_speed,
            )

            # Bo dau # de kiem tra ket qua phan loai
            # va cach tinh duration.
            #
            # words = extract_words(sentence)
            #
            # simple_unmarked = [
            #     word
            #     for word_index, word in enumerate(words)
            #     if is_common_unmarked_vietnamese_word(
            #         word,
            #         word_index,
            #     )
            # ]
            #
            # japanese_romaji_words = [
            #     word
            #     for word in words
            #     if is_japanese_romaji_word(word)
            # ]
            #
            # print(
            #     f"\nCau {idx}/{len(sentences)}: "
            #     f"{sentence}"
            # )
            #
            # print(
            #     "Tu Viet khong dau thong dung:",
            #     simple_unmarked,
            # )
            #
            # print(
            #     "Ten Nhat romaji:",
            #     japanese_romaji_words,
            # )
            #
            # print(
            #     "So tu Nhat romaji:",
            #     count_japanese_romaji_words(sentence),
            # )
            #
            # print(
            #     "Duration bo sung cho romaji:",
            #     round(
            #         estimate_japanese_romaji_buffer(sentence),
            #         3,
            #     ),
            # )
            #
            # print(
            #     "So tu phuc tap:",
            #     count_likely_complex_words(sentence),
            # )
            #
            # print(
            #     "Thoi luong generate:",
            #     duration,
            # )

            audio = model.generate(
                text=sentence,
                ref_audio=ref_audio_path,
                ref_text=ref_text,
                duration=duration,
                language="vi",
            )

            segment = normalize_audio_array(
                audio[0]
            )

            segment = speedup_audio_segment(
                segment,
                speed=segment_speed,
                sample_rate=SAMPLE_RATE,
            )

            all_audio_segments.append(segment)

            if idx < len(sentences):
                all_audio_segments.append(
                    make_silence(
                        nghi_cau,
                        SAMPLE_RATE,
                    )
                )

        if not all_audio_segments:
            print("Khong co audio de ghep.")
            return

        final_audio = np.concatenate(
            all_audio_segments
        )

        sf.write(
            "clone_out.wav",
            final_audio,
            SAMPLE_RATE,
        )

        display(
            Audio(
                "clone_out.wav",
                autoplay=False,
            )
        )

        print("\nXong.")
        print("File hoan chinh: clone_out.wav")
        print(
            f"Moi doan da tang toc: "
            f"{segment_speed}x"
        )
        print(
            f"Khoang nghi giu nguyen: "
            f"{nghi_cau}s"
        )


# =========================
# Dieu khien khoa va mo khoa
# =========================

UNLOCK_JS = r"""
(function () {
    window.omnivoiceScreenLocked = false;

    if (window.omnivoiceLockInterval) {
        clearInterval(window.omnivoiceLockInterval);
        window.omnivoiceLockInterval = null;
    }

    if (window.omnivoiceLockObserver) {
        window.omnivoiceLockObserver.disconnect();
        window.omnivoiceLockObserver = null;
    }

    if (window.omnivoiceLockScrollHandler) {
        window.removeEventListener(
            "scroll",
            window.omnivoiceLockScrollHandler,
            true
        );

        window.omnivoiceLockScrollHandler = null;
    }

    if (window.omnivoiceLockResizeHandler) {
        window.removeEventListener(
            "resize",
            window.omnivoiceLockResizeHandler
        );

        window.omnivoiceLockResizeHandler = null;
    }
})();
"""

LOCK_JS = r"""
(function () {
    const anchorId = "omnivoice-input-anchor";

    if (window.omnivoiceLockInterval) {
        clearInterval(window.omnivoiceLockInterval);
    }

    if (window.omnivoiceLockObserver) {
        window.omnivoiceLockObserver.disconnect();
    }

    if (window.omnivoiceLockScrollHandler) {
        window.removeEventListener(
            "scroll",
            window.omnivoiceLockScrollHandler,
            true
        );
    }

    if (window.omnivoiceLockResizeHandler) {
        window.removeEventListener(
            "resize",
            window.omnivoiceLockResizeHandler
        );
    }

    window.omnivoiceScreenLocked = true;

    function findAnchor() {
        return document.getElementById(anchorId);
    }

    function lockScreenToInput() {
        if (!window.omnivoiceScreenLocked) {
            return;
        }

        const anchor = findAnchor();

        if (!anchor) {
            return;
        }

        anchor.scrollIntoView({
            behavior: "auto",
            block: "start",
            inline: "nearest"
        });
    }

    lockScreenToInput();

    [30, 60, 100, 150, 250, 400, 700, 1000].forEach(
        function (delay) {
            setTimeout(lockScreenToInput, delay);
        }
    );

    window.omnivoiceLockInterval = setInterval(
        lockScreenToInput,
        100
    );

    window.omnivoiceLockObserver = new MutationObserver(
        function () {
            if (window.omnivoiceScreenLocked) {
                requestAnimationFrame(lockScreenToInput);
            }
        }
    );

    window.omnivoiceLockObserver.observe(
        document.body,
        {
            childList: true,
            subtree: true
        }
    );

    window.omnivoiceLockScrollHandler = function () {
        if (window.omnivoiceScreenLocked) {
            requestAnimationFrame(lockScreenToInput);
        }
    };

    window.addEventListener(
        "scroll",
        window.omnivoiceLockScrollHandler,
        true
    );

    window.omnivoiceLockResizeHandler = function () {
        if (window.omnivoiceScreenLocked) {
            requestAnimationFrame(lockScreenToInput);
        }
    };

    window.addEventListener(
        "resize",
        window.omnivoiceLockResizeHandler
    );
})();
"""


def run_unlock_javascript():
    """Tat toan bo co che khoa cuon."""
    display(
        Javascript(UNLOCK_JS)
    )


def run_lock_javascript():
    """Bat lai co che khoa cuon."""
    display(
        Javascript(LOCK_JS)
    )


def toggle_scroll_lock(button):
    """Chuyen doi giua khoa va mo khoa cuon."""
    if scroll_lock_state["locked"]:
        run_unlock_javascript()

        scroll_lock_state["locked"] = False

        button.description = "Khoa cuon"
        button.icon = "lock"
        button.button_style = "warning"
        button.tooltip = (
            "Khoa man hinh tro lai tai khung nhap"
        )

        lock_status.value = (
            "<span style='color:#2e7d32; font-weight:600;'>"
            "Da mo khoa. Ban co the cuon man hinh binh thuong."
            "</span>"
        )

    else:
        run_lock_javascript()

        scroll_lock_state["locked"] = True

        button.description = "Mo khoa cuon"
        button.icon = "unlock"
        button.button_style = "info"
        button.tooltip = (
            "Mo khoa de cuon man hinh binh thuong"
        )

        lock_status.value = (
            "<span style='color:#d32f2f; font-weight:600;'>"
            "Man hinh dang duoc khoa tai khung nhap."
            "</span>"
        )


text_box.observe(
    on_text_change,
    names="value",
)

scroll_lock_button.on_click(
    toggle_scroll_lock
)

control_row = widgets.HBox(
    [
        scroll_lock_button,
        lock_status,
    ],
    layout=widgets.Layout(
        align_items="center",
        gap="12px",
        margin="8px 0 8px 50px",
    ),
)

display(
    lock_anchor,
    text_box,
    control_row,
    output,
)

# Tu dong khoa khi cell duoc chay.
display(
    Javascript(LOCK_JS)
)